In [ ]:
# bootstrap: Colab clone + local import of `gpusim` (auto-inserted)
import sys, pathlib
if "google.colab" in sys.modules:
    import os, subprocess
    _slug = "aniryou/full-stack-agentic-engineer"
    _repo = pathlib.Path("/content/full-stack-agentic-engineer")
    if not _repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_repo)], check=True)
    os.chdir(_repo / "02-cuda-nccl-runtime/cuda-and-nccl/cuda-nccl-core")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
_r = pathlib.Path.cwd().resolve()
while _r != _r.parent and not (_r / "gpusim").exists():
    _r = _r.parent
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))
del _r

# 02 · Tiling, fusion, launches and occupancy

**Tier:** T0 (CPU, simulated, numpy only). The lab's `02_memory_bound_kernels_on_a_real_gpu` (T1)
has real timings of the same kernels.

## The one-minute version
The time of a memory-bound kernel is *(bytes it moves) / (memory bandwidth)*, plus its launch.
There are four levers:

* **Tiling.** A GEMM stages $\mathit{BM} \times \mathit{BK}$ and $\mathit{BK} \times \mathit{BN}$ tiles in
  shared memory. Then it reloads $A$ only $\lceil N/\mathit{BN} \rceil$ times and $B$ only $\lceil M/\mathit{BM} \rceil$
  times, and not $N$ and $M$ times. Traffic decreases by approximately the tile size.
* **Fusion.** A softmax made of five kernels makes approximately eight passes of matrix size
  through HBM. A fused softmax makes two passes. It makes three with online softmax, when a row
  does not fit on chip. FlashAttention is this idea, applied to attention.
* **CUDA Graphs.** When kernels are shorter than their launch cost, the GPU waits for the CPU. A
  graph replays the full sequence with one launch. This is why engines capture decode steps.
* **Occupancy.** Registers, shared memory, and warp and block slots set a limit on the number of
  warps that an SM keeps resident. You need sufficient warps, or sufficient loads in flight per
  warp, to cover memory latency (Little's law). More is not always better.

Primer: §3 *Memory access patterns*, §2 *The execution model* (occupancy) and §4 *Streams,
launch overhead and CUDA Graphs* (`../../PRIMER.md`).

In [ ]:
import math

import numpy as np

from gpusim import occupancy as occ
from gpusim import tiling

n = 4096                                               # C = A @ B, all 4096 x 4096, bf16
print(f"{'variant':>18}  {'global traffic':>14}  {'FLOP/byte':>9}  {'time if all from HBM on H100':>28}")
for label, (bm, bn) in [("naive (1x1)", (1, 1)), ("tile 32x32", (32, 32)), ("tile 128x128", (128, 128)),
                        ("tile 128x256", (128, 256)), ("compulsory", (None, None))]:
    t = tiling.gemm_traffic(n, n, n, bm, bn, dtype_bytes=2)
    print(f"{label:>18}  {t['bytes'] / 1e9:>11.2f} GB  {t['intensity']:>9.1f}  {t['bytes'] / 3.35e12 * 1e3:>25.2f} ms")

The $4096^3$ GEMM is 137 GFLOP. That is approximately 0.14 ms at ~990 TFLOP/s of dense BF16. If
the GEMM has no reuse, it streams 275 GB, which takes 82 ms at H100 bandwidth. A 128x128
shared-memory tile decreases global traffic by approximately 126x.

**L2** removes the rest of the gap to the compulsory 0.1 GB. Many thread blocks reuse the same
$A$ and $B$ panels while those panels are still in the 50 MB L2. Thus HBM sees much less than the
"global traffic" column. The roofline and the ridge point are the topic of layer 01. Here we
count bytes.

The simulator runs a real tiled GEMM on the CPU and counts its global loads:

In [ ]:
rng = np.random.default_rng(0)
A = rng.integers(-4, 5, (96, 80)).astype(np.float64)
B = rng.integers(-4, 5, (80, 112)).astype(np.float64)
C, counted = tiling.tiled_matmul(A, B, BM=32, BN=32, BK=16)
print("exact:", np.array_equal(C, A @ B), "| counted:", counted,
      "| formula:", {k: v for k, v in tiling.gemm_traffic(96, 112, 80, 32, 32).items() if k.endswith("elems")})

## Exercise 2.1: the tiled-GEMM traffic formula

Write `tiled_traffic_bytes(M, N, K, BM, BN, dtype_bytes)`. Each output tile loads its full
$\mathit{BM} \times K$ panel of $A$ and $K \times \mathit{BN}$ panel of $B$. It writes its
$\mathit{BM} \times \mathit{BN}$ block of $C$ one time. Tiles at the edges are partial, thus count only real
elements. Hint: the kernel loads each element of $A$ one time for each *column* of tiles.

In [ ]:
def tiled_traffic_bytes(M, N, K, BM, BN, dtype_bytes=2):
    a_loads = M * K * math.ceil(N / BN)
    b_loads = K * N * math.ceil(M / BM)
    return (a_loads + b_loads + M * N) * dtype_bytes

In [ ]:
for shape in [(4096, 4096, 4096, 128, 128), (96, 112, 80, 32, 32), (1000, 300, 70, 64, 128), (8, 4096, 4096, 64, 64)]:
    assert tiled_traffic_bytes(*shape) == tiling.gemm_traffic(*shape)["bytes"], shape
_, c = tiling.tiled_matmul(A, B, 32, 32, 16)
assert tiled_traffic_bytes(96, 112, 80, 32, 32, 8) == (c["loads"] + c["stores"]) * 8
print("✅ traffic = (M*K*ceil(N/BN) + K*N*ceil(M/BM) + M*N) x bytes, matching the simulated kernel")

Look at the last shape, $M = 8$: a decode-sized GEMM with 8 tokens against a 4096x4096 weight.
Most of its traffic is one read of $B$ (the weights). That is the batch-1 decode problem
in one line: no tile size can increase the intensity above the limit that $M$ sets.

## Tiles cost shared memory, and shared memory caps occupancy
A block that computes a $\mathit{BM} \times \mathit{BN}$ tile stages
$(\mathit{BM} \cdot \mathit{BK} + \mathit{BK} \cdot \mathit{BN})$ elements per pipeline stage:

In [ ]:
smem = tiling.tile_smem_bytes(128, 128, 32, dtype_bytes=2, stages=3)
for cc in ("8.9", "9.0"):
    o = occ.occupancy(256, regs_per_thread=128, smem_per_block=smem, cc=cc)
    print(f"CC {cc} ({occ.SMS[cc].name}): 128x128x32 bf16, 3 stages = {smem // 1024} KB/block -> "
          f"{o.blocks_per_sm} blocks/SM, {o.active_warps} warps, occupancy {o.occupancy:.0%}, limits {o.limits}")

## Exercise 2.2: the register limit, by hand

Registers are the most frequent limit on occupancy. On CC 7.x to 10.x, an SM has 65,536 32-bit
registers, divided across **4 sub-partitions** of 16,384. The hardware allocates registers **per
warp**, in units of 256 registers. Write `blocks_by_registers(threads_per_block, regs_per_thread)`
with these rules (assume $\mathtt{regs\_per\_thread} \le 255$):

1. registers per warp $= \mathtt{regs\_per\_thread} \times 32$, rounded up to a multiple of 256
2. warps per sub-partition $= \lfloor 16384 / \text{registers per warp} \rfloor$, and warps per
   SM $= 4 \times$ that
3. blocks $= \lfloor \text{warps per SM} / \lceil \mathtt{threads\_per\_block} / 32 \rceil \rfloor$

In [ ]:
def blocks_by_registers(threads_per_block, regs_per_thread):
    per_warp = math.ceil(regs_per_thread * 32 / 256) * 256
    warps_per_sm = (16384 // per_warp) * 4
    return warps_per_sm // math.ceil(threads_per_block / 32)

In [ ]:
for threads in (32, 64, 96, 128, 256, 512, 1024):
    for regs in (16, 24, 32, 40, 64, 72, 80, 96, 128, 168, 255):
        want = occ.occupancy(threads, regs_per_thread=regs, cc="9.0").limits["registers"]
        assert blocks_by_registers(threads, regs) == want, (threads, regs, want)
print("✅ blocks_by_registers matches the occupancy calculator for 77 configurations")
print("   e.g. 80 regs, 32 threads:", blocks_by_registers(32, 80), "blocks (not 65536 // 2560 = 25)")

## Exercise 2.3: pick a GEMM tile for an L4

You select a tile for a $4096^3$ BF16 GEMM on an L4 (CC 8.9). A block has 256 threads, and each
thread uses 128 registers. You want the **highest arithmetic intensity** (from
`tiling.gemm_traffic`) of the candidates that still fit **at least 2 blocks per SM**
(use `occ.occupancy` with `tiling.tile_smem_bytes`). Two resident blocks let the loads of one
block overlap the math of the other. Write `choose_tile(candidates)`. It returns the best
`(BM, BN, BK, stages)`.

In [ ]:
candidates = [(64, 64, 32, 2), (128, 128, 32, 3), (128, 256, 32, 3), (256, 128, 64, 2),
              (128, 128, 64, 4), (128, 256, 16, 2)]

def choose_tile(candidates):
    def fits(c):
        bm, bn, bk, st = c
        smem_bytes = tiling.tile_smem_bytes(bm, bn, bk, 2, st)
        return occ.occupancy(256, regs_per_thread=128, smem_per_block=smem_bytes, cc="8.9").blocks_per_sm >= 2
    ok = [c for c in candidates if fits(c)]
    return max(ok, key=lambda c: tiling.gemm_traffic(4096, 4096, 4096, c[0], c[1])["intensity"])

In [ ]:
best = choose_tile(candidates)
assert best == (128, 256, 16, 2), best
print("✅", best, "wins: BM x BN sets the intensity, while BK and stages only set the shared memory,")
print("   so a thin BK buys the big output tile. (A real kernel also pays registers for the 128x256")
print("   accumulator, which is why production GEMMs often run 1 block/SM and hide latency by pipelining.)")

## Fusion: the bytes between kernels
The next cell calculates the traffic of a row-wise softmax over a 4096x4096 BF16 score matrix.
That is the size of the scores of one attention head at 4k context:

In [ ]:
R = Cc = 4096
for v in ("unfused", "fused", "online"):
    t = tiling.softmax_traffic(R, Cc, 2, v)
    print(f"{v:>8}: {t['kernels']} kernel(s), {t['bytes'] / 2**20:>6.0f} MiB of HBM traffic, "
          f"{t['bytes'] / 3.35e12 * 1e6:>5.1f} us at 3.35 TB/s (simulated arithmetic)")

Unfused, each step (max, subtract, exp, sum, divide) reads its input from HBM and writes its output
back. That is approximately 4x the traffic of a fused kernel, plus five launches and not one.

When a row is too long to hold on chip, the **online** version streams the row one time. During
that pass, it keeps the current max $m$ and the current sum $s$. When the max increases, it
rescales $s$ by $\exp(m_{\text{old}} - m_{\text{new}})$. Then it streams the row again to write the
output. FlashAttention does one more step: it fuses the softmax *into* the $QK^{\top}$ and ${PV}$
matmuls. Thus the score matrix never gets to HBM at all.

## Exercise 2.4: online softmax

Write `online_softmax(x, block)` for a 2-D array. Walk the columns `block` at a time, never the
full row. During the walk, keep only the current max and the current sum per row. Then make the
output. The output must match `tiling.softmax_ref`. The function must not overflow on large inputs.

In [ ]:
def online_softmax(x, block=128):
    x = np.asarray(x, dtype=np.float64)
    m = np.full((x.shape[0], 1), -np.inf)
    s = np.zeros((x.shape[0], 1))
    for c in range(0, x.shape[1], block):
        blk = x[:, c:c + block]
        m_new = np.maximum(m, blk.max(axis=1, keepdims=True))
        s = s * np.exp(m - m_new) + np.exp(blk - m_new).sum(axis=1, keepdims=True)
        m = m_new
    return np.exp(x - m) / s

In [ ]:
x = rng.normal(size=(6, 1000)) * 400                    # exp(x) overflows float64 once x > 709.8
assert x.max() > 710                                    # so a softmax that skips the max would overflow here
y = online_softmax(x, block=64)
assert np.allclose(y, tiling.softmax_ref(x), rtol=1e-12, atol=0) and np.all(np.isfinite(y))
assert np.allclose(online_softmax(x, block=7), y, rtol=1e-12)
print("✅ online softmax is exact for any block size: rescaling by exp(m_old - m_new) keeps the books balanced")

## Launch overhead and CUDA Graphs
At small batch, a decode step is hundreds of short kernels. Each eager launch costs CPU time
(driver plus framework dispatch). If a kernel finishes before the CPU enqueues the next kernel,
the GPU is idle. The numbers in the next cell are **assumptions** (5 us per launch, 10 us to launch
a graph). The lab measures them on a real GPU.

In [ ]:
for kernel_us in (2.0, 5.0, 20.0):
    ks = [kernel_us] * 384                               # e.g. 32 layers x 12 kernels
    e, g = tiling.step_time(ks, launch_us=5.0), tiling.step_time(ks, launch_us=5.0, graph=True)
    print(f"kernels of {kernel_us:>4} us: eager {e['time_us']:>6.0f} us (GPU idle {e['gpu_idle']:.0%}), "
          f"graph {g['time_us']:>6.0f} us -> {e['time_us'] / g['time_us']:.2f}x")

## Exercise 2.5: is this decode step launch-bound?

A decode step at batch 8 runs 40 layers x 10 kernels. Each kernel takes **3 us** on the GPU. Eager
PyTorch uses **12 us** of CPU time per launch (assumed). A captured graph costs one **15 us**
launch. Predict these values, *without* a call to `step_time`:

* `eager_us`: the eager step time
* `graph_us`: the graph step time
* `breakeven_kernel_us`: the kernel duration above which the eager step becomes GPU-bound. Above
  that duration, graphs save one launch at the most.

In [ ]:
n_kernels = 40 * 10
eager_us = n_kernels * 12 + 3          # the CPU is the bottleneck: the last launch plus one kernel
graph_us = 15 + n_kernels * 3          # one launch, then back to back
breakeven_kernel_us = 12               # kernels longer than a launch keep the GPU fed

In [ ]:
ks = [3.0] * 400
assert eager_us == tiling.step_time(ks, launch_us=12)["time_us"]
assert graph_us == tiling.step_time(ks, launch_us=12, graph=True, graph_launch_us=15)["time_us"]
assert tiling.step_time([breakeven_kernel_us + 0.5] * 400, 12)["gpu_idle"] < 0.01
assert tiling.step_time([breakeven_kernel_us - 0.5] * 400, 12)["gpu_idle"] > 0.03
print(f"✅ eager {eager_us:.0f} us vs graph {graph_us:.0f} us ({eager_us / graph_us:.1f}x): "
      "graphs pay off when kernels are shorter than launches")

## Occupancy is a means: Little's law
To keep bandwidth $B$ with latency $L$, $B \times L$ bytes must be in flight at each instant. With an
**assumed** 600 ns loaded DRAM latency, compare the number of independent 4-byte loads that each
resident thread must keep in flight:

In [ ]:
for dev in ("L4", "A100-80GB", "H100-SXM"):
    d = occ.DEVICES[dev]
    per_sm = occ.bytes_in_flight(d["hbm_Bps"], 600e-9) / d["sms"]
    print(f"{dev:>9}: {per_sm / 1024:5.1f} KB in flight per SM -> "
          f"{occ.loads_in_flight_per_thread(dev, 600e-9):.2f} x 4-byte loads/thread at full occupancy, "
          f"{occ.loads_in_flight_per_thread(dev, 600e-9, bytes_per_load=16):.2f} x 16-byte loads")
print("tail effect:", occ.waves(140, n_sms=132, blocks_per_sm=1), "<- 140 blocks on 132 SMs")

An L4 saturates its 300 GB/s when half of its threads each hold one load. An H100 needs almost two
4-byte loads in flight per thread, even at full occupancy. Thus fast kernels use one or more of these:

* 16-byte vector loads
* several independent loads per thread (ILP)
* asynchronous copies (cp.async, TMA) that do not occupy threads at all

That is why optimized GEMM and attention kernels run at *low* occupancy and still saturate the
machine.

## In a design review

**The two-minute version.** "Our kernels are memory-bound, thus we count bytes. Tiling makes each
byte that comes from HBM feed many FLOPs from shared memory and registers. A 128-wide tile
decreases GEMM traffic by approximately 128x, and L2 captures the reuse between tiles. Fusion
removes the round trips between kernels.

"A fused softmax makes 2 HBM passes per element. An unfused softmax makes approximately 8.
FlashAttention never writes the score matrix at all. At small batch, decode is hundreds of
microsecond-scale kernels.
Thus the CPU launch path becomes the bottleneck, and we capture CUDA graphs per batch size.

"We set occupancy with Little's law, not as a target. We want sufficient bytes in flight to cover
latency, from warps, from ILP or from async copies."

**Drill questions**

1. *FlashAttention does the same FLOPs. Why is it 2-4x faster?* Attention was memory-bound,
   because the $N \times N$ score matrix went to HBM and back. A fusion of $QK^{\top}$, softmax and
   ${PV}$ into one tiled kernel removes that traffic. Thus the time decreases to near the cost of
   the matmuls alone.
2. *The profile shows gaps between small kernels in decode. What do you try?* Try CUDA graphs
   (capture one graph per batch-size bucket). Fuse the elementwise ops (torch.compile). Remove the
   host syncs (`.item()`, prints) from the step.
3. *Is it correct to push occupancy to 100%?* Only if nothing covers the latency. Large-tile GEMMs
   give up occupancy to get register reuse, and they hide latency with pipelined async copies.
   Examine the achieved bandwidth and the stall reasons, not the occupancy number.